## STEP 0. 제출용 Notebook과 Chapter10 입력 준비

### 설명
이번 실습에서는 data/processed 폴더에 있는 전처리 완료 데이터를 사용합니다.

현재 Notebook 위치가 notebooks/ch10/00.ipynb이므로, 데이터 폴더까지는 ../../data/processed 경로로 접근하면 됩니다.

In [6]:
from pathlib import Path
import pandas as pd

# 데이터 폴더 위치
DATA_DIR = Path("../../data/processed")

# CSV 파일을 DataFrame으로 불러오기
customers = pd.read_csv(DATA_DIR / "customers_clean.csv")
orders = pd.read_csv(DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(DATA_DIR / "order_items_clean.csv")
products = pd.read_csv(DATA_DIR / "products_clean.csv")

# orders가 제대로 불러와졌는지 확인
print(type(orders))
print(orders.columns)

<class 'pandas.DataFrame'>
Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status', 'order_month', 'order_dayofweek'],
      dtype='str')


### 결과 해석
각 데이터의 (행 수, 열 수)가 출력되면 정상적으로 불러온 것입니다.
예를 들어 orders: (1000, 8)이면 orders 데이터는 1000행, 8열이라는 뜻입니다.

## STEP 1. Target Contract와 예측 시점 정의

### 설명
이번 실습에서는 주문 상태 중 **`completed`와 `cancelled`만 사용**합니다.

- `completed` → `0`
- `cancelled` → `1`
- `refunded` 등 다른 상태 → 제외

예측 시점은 **주문이 생성된 직후**입니다. 따라서 주문이 취소된 뒤에 생기는 정보는 모델 입력으로 사용하면 안 됩니다. :chatgpt-content-reference{index="0"}

### 코드 작성

In [7]:
# completed와 cancelled 주문만 선택
model_orders = orders[
    orders["order_status"].isin(["completed", "cancelled"])
].copy()

# Target 생성
model_orders["is_cancelled"] = model_orders["order_status"].map({
    "completed": 0,
    "cancelled": 1
})

# 결과 확인
print(model_orders["order_status"].value_counts())
print()
print(model_orders["is_cancelled"].value_counts())

order_status
completed    184
cancelled     64
Name: count, dtype: int64

is_cancelled
0    184
1     64
Name: count, dtype: int64


### 결과 해석
`order_status`에서는 `completed`, `cancelled`만 남아 있어야 합니다.

`is_cancelled`에서는

```text
0 = 정상 완료 주문
1 = 취소 주문
```

으로 변환된 것을 확인하면 됩니다.

이제부터 모델은 **`is_cancelled`가 0인지 1인지 예측**하게 됩니다.

## STEP 2. 주문 단위 특징과 계산 관계 검증

### 설명
예측 단위는 **주문 1건**이므로 `order_items` 데이터를 `order_id` 기준으로 묶어 주문별 특징을 만듭니다.

먼저 아래 관계가 맞는지 확인합니다.

`line_total = quantity × unit_price`

그다음 주문별로 다음 값을 만듭니다.

- `item_count` : 상품 종류 수
- `total_quantity` : 총 주문 수량
- `order_amount` : 총 주문 금액 :chatgpt-content-reference{index="0"}

### 코드 작성

In [8]:
# line_total 계산이 맞는지 확인
calculated_line_total = order_items["quantity"] * order_items["unit_price"]

line_total_check = (
    calculated_line_total.round(2) == order_items["line_total"].round(2)
)

print("line_total 계산 일치:", line_total_check.all())

# 주문별 특징 생성
order_features = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("product_id", "count"),
        total_quantity=("quantity", "sum"),
        order_amount=("line_total", "sum")
    )
    .reset_index()
)

# 결과 확인
print(order_features.head())
print()
print("주문 수:", len(order_features))

line_total 계산 일치: True
   order_id  item_count  total_quantity  order_amount
0         1           4              14       1436000
1         2           1               4        756000
2         3           3              13       1435000
3         4           2               7        498000
4         5           2               6        941000

주문 수: 300


### 결과 해석

line_total 계산 일치: True이므로 모든 상품 행에서 quantity × unit_price = line_total 관계가 정상입니다.

또한 주문이 총 300건이고, order_features가 주문별로 1행씩 만들어졌습니다. 따라서 주문 단위 특징 생성까지 정상 완료입니다.

## STEP 3. 병합 관계와 Feature Audit 확인

### 설명
방금 만든 주문별 특징을 `orders`와 합칩니다.

이때 중요한 것은 **병합하면서 주문 수가 이상하게 늘어나거나 줄어들지 않는지** 확인하는 것입니다. 이후 모델에 사용하면 안 되는 컬럼도 확인합니다. :chatgpt-content-reference{index="1"}

### 코드 작성

In [9]:
# 병합 전 주문 수
before_rows = len(model_orders)

# 주문 데이터 + 주문별 특징 병합
model_data = model_orders.merge(
    order_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# 병합 후 주문 수
after_rows = len(model_data)

# 주문 특징이 없는 데이터 확인
unmatched_count = model_data["order_amount"].isna().sum()

print("병합 전 주문 수:", before_rows)
print("병합 후 주문 수:", after_rows)
print("미매칭 주문 수:", unmatched_count)

# 모델 입력에 사용하면 안 되는 컬럼
forbidden_features = [
    "order_status",
    "is_cancelled",
    "order_id",
    "customer_id",
    "product_id"
]

# 현재 데이터에 존재하는 금지 컬럼 확인
found_forbidden = [
    col for col in forbidden_features
    if col in model_data.columns
]

print("확인된 금지 컬럼:", found_forbidden)

병합 전 주문 수: 248
병합 후 주문 수: 248
미매칭 주문 수: 0
확인된 금지 컬럼: ['order_status', 'is_cancelled', 'order_id', 'customer_id']


### 결과 해석
병합 전과 후의 주문 수가 같고, 병합 전후 모두 248건이므로 주문 수가 변하지 않았습니다.

미매칭 주문 수: 0이므로 모든 주문에 주문별 특징이 정상적으로 연결되었습니다.

order_status, is_cancelled, order_id, customer_id는 데이터에 존재하지만 모델 입력 Feature에서는 제외해야 합니다.

처음 주문 특징은 300건이었지만 현재 248건인 이유는 STEP 1에서 completed, cancelled만 남기고 다른 주문 상태를 제외했기 때문입니다.

## STEP 4. 클래스 분포와 Dummy Baseline 확인

### 설명
먼저 completed(0)와 cancelled(1)가 각각 얼마나 있는지 확인합니다.

그다음 가장 많은 클래스만 무조건 예측하는 Dummy 모델을 만들어 실제 모델이 최소한 이 기준보다 나은지 비교할 준비를 합니다.

### 코드 작성

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

# 클래스 개수와 비율 확인
print("클래스 개수")
print(model_data["is_cancelled"].value_counts())

print("\n클래스 비율")
print(model_data["is_cancelled"].value_counts(normalize=True).round(3))

# 임시로 Target만 나누기
X = model_data.drop(columns=["is_cancelled"])
y = model_data["is_cancelled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 가장 많은 클래스만 예측하는 Dummy 모델
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

print("\nDummy Accuracy:", round(accuracy_score(y_test, dummy_pred), 3))
print("Dummy Cancelled Recall:", round(recall_score(y_test, dummy_pred), 3))

클래스 개수
is_cancelled
0    184
1     64
Name: count, dtype: int64

클래스 비율
is_cancelled
0    0.742
1    0.258
Name: proportion, dtype: float64

Dummy Accuracy: 0.74
Dummy Cancelled Recall: 0.0


### 결과 해석
전체 248건 중
- completed(0) : 184건, 약 74.2%
- cancelled(1) : 64건, 약 25.8%

으로 취소 주문이 더 적어서 클래스 불균형이 조금 있습니다.

Dummy 모델의 Accuracy는 0.74로 높아 보이지만, Cancelled Recall = 0.0입니다.

즉, Dummy 모델은 사실상 모든 주문을 completed라고 예측해서 정확도만 높게 나온 것입니다. 취소 주문은 하나도 찾아내지 못합니다.

따라서 이후 실제 모델은 Accuracy만 보지 말고 Precision, Recall, F1도 함께 확인해야 합니다.

### STEP 5. Train / Validation / Test 분리

## 설명

데이터를 세 부분으로 나눕니다.
- Train : 모델 학습
- Validation : 모델과 threshold 선택
- Test : 마지막 최종 평가

Test 데이터는 마지막까지 모델 선택에 사용하지 않습니다.

## 코드 작성

In [13]:
from sklearn.model_selection import train_test_split

# 입력값과 Target 분리
X = model_data.drop(columns=["is_cancelled"])
y = model_data["is_cancelled"]

# 1차 분리: Train 60%, 나머지 40%
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

# 2차 분리: Validation 20%, Test 20%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

print("\nTrain 클래스")
print(y_train.value_counts())

print("\nValidation 클래스")
print(y_val.value_counts())

print("\nTest 클래스")
print(y_test.value_counts())

Train: 148
Validation: 50
Test: 50

Train 클래스
is_cancelled
0    110
1     38
Name: count, dtype: int64

Validation 클래스
is_cancelled
0    37
1    13
Name: count, dtype: int64

Test 클래스
is_cancelled
0    37
1    13
Name: count, dtype: int64


### 결과 해석

데이터가 정상적으로 Train 148 / Validation 50 / Test 50으로 나뉘었습니다.

또한 세 데이터 모두 0과 1이 존재하므로 정상입니다.
- Train → 모델 학습
- Validation → 모델과 Threshold 선택
- Test → 마지막 최종 평가

Test는 마지막 단계까지 건드리지 않습니다.

## STEP 5-1. 모델 입력 컬럼 확인

### 설명

실제 모델에 넣을 Feature를 정하기 전에 현재 데이터에 어떤 컬럼이 있는지 확인합니다.

In [14]:
# 현재 주문 데이터 컬럼 확인
print("model_data 컬럼")
print(model_data.columns.tolist())

print("\ncustomers 컬럼")
print(customers.columns.tolist())

model_data 컬럼
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek', 'is_cancelled', 'item_count', 'total_quantity', 'order_amount']

customers 컬럼
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']


## STEP 5-2. 사용할 Feature 선택

## 설명

현재 컬럼을 확인했으니 이제 모델에 실제로 넣을 Feature만 선택합니다.

고객 정보에서 gender, age, city를 가져오고, 최종적으로 아래 6개만 사용하겠습니다.

- order_month
- order_dayofweek
- age
- payment_method
- gender
- city

order_id, customer_id, order_status 같은 식별자·결과 정보는 모델에서 제외합니다.

### 코드 작성

In [15]:
# 고객 정보 추가
model_data = model_data.merge(
    customers[["customer_id", "gender", "age", "city"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# 모델에 사용할 Feature
features = [
    "order_month",
    "order_dayofweek",
    "age",
    "payment_method",
    "gender",
    "city"
]

# 입력 데이터와 Target
X = model_data[features]
y = model_data["is_cancelled"]

print("사용 Feature:")
print(X.columns.tolist())

print("\n데이터 크기:", X.shape)
print("\n결측치:")
print(X.isna().sum())

사용 Feature:
['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']

데이터 크기: (248, 6)

결측치:
order_month        0
order_dayofweek    0
age                0
payment_method     0
gender             0
city               0
dtype: int64


### 결과 해석
모델 학습에 사용할 입력 데이터가 정상적으로 준비되었습니다.

식별자나 결과 정보는 제외했고, 사용할 Feature 6개에는 결측치가 없으므로 다음 단계인 Pipeline 전처리로 진행할 수 있습니다.

## STEP 6. Pipeline 전처리 확인

### 설명

숫자형과 범주형 Feature는 처리 방법이 다릅니다.
- 숫자형 → 결측치를 중앙값으로 채우고 StandardScaler
- 범주형 → 결측치를 최빈값으로 채우고 OneHotEncoder
- 실제 전처리는 Pipeline 안에서 Train 데이터로만 학습합니다.

### 코드 작성

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 숫자형 Feature
numeric_features = [
    "age"
]

# 범주형 Feature
categorical_features = [
    "order_month",
    "order_dayofweek",
    "payment_method",
    "gender",
    "city"
]

# 숫자형 전처리
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 범주형 전처리
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 전처리 합치기
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("숫자형 Feature:", numeric_features)
print("범주형 Feature:", categorical_features)
print("Pipeline 전처리 준비 완료")

숫자형 Feature: ['age']
범주형 Feature: ['order_month', 'order_dayofweek', 'payment_method', 'gender', 'city']
Pipeline 전처리 준비 완료


### 결과 해석

숫자형 Feature는 age 1개, 범주형 Feature는 5개로 정상적으로 구분되었습니다.

order_month도 범주형으로 처리되었기 때문에 문자열 변환 오류가 해결되었습니다.

## STEP 7. Validation에서 후보 모델 비교

### 설명

이제 같은 전처리 Pipeline을 사용해서 다음 3개 모델을 비교합니다.
- Dummy : 기준 모델
- Logistic Regression
- Random Forest

Validation 데이터에서 Accuracy, Precision, Recall, F1을 비교합니다. Test 데이터는 아직 사용하지 않습니다.

### 코드 작성

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 현재 6개 Feature 기준으로 다시 분리
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# 비교할 모델
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

results = []

# 각 모델 학습 및 Validation 평가
for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1": f1_score(y_val, y_pred, zero_division=0)
    })

# 결과 표
validation_results = pd.DataFrame(results)

display(validation_results.round(3))

,Model,Accuracy,Precision,Recall,F1
0,Dummy,0.74,0.000,0.000,0.000
1,Logistic Regression,0.70,0.375,0.231,0.286
2,Random Forest,0.72,0.333,0.077,0.125


### 결과 해석

Dummy 모델은 Accuracy가 **0.74**로 가장 높지만, Precision·Recall·F1이 모두 **0**입니다. 즉, 취소 주문을 전혀 찾아내지 못합니다.

Logistic Regression은 Accuracy는 **0.70**이지만 Recall이 **0.231**, F1이 **0.286**으로 세 후보 중 가장 높습니다.

Random Forest는 Accuracy가 **0.72**이지만 Recall이 **0.077**로 매우 낮아 취소 주문 탐지 성능이 떨어집니다.

따라서 Validation 기준에서는 **Logistic Regression이 취소 주문을 가장 잘 탐지하는 후보**로 볼 수 있습니다. 다음 단계에서는 이 모델의 Threshold를 조정해 성능 변화를 확인하면 됩니다.

## STEP 8. Validation에서 Threshold 선택

### 설명

STEP 7에서 선택한 Logistic Regression의 예측 확률을 이용해 Threshold를 비교합니다.

Threshold는 몇 % 이상이면 취소 주문으로 판단할지 정하는 기준값입니다.

- Threshold를 낮추면 → 취소를 더 많이 잡지만 FP가 늘 수 있음
- Threshold를 높이면 → 취소 예측은 줄지만 FN이 늘 수 있음

이번에는 Validation 데이터에서 F1 → Recall → Precision 순으로 좋은 Threshold를 선택합니다.

### 코드 작성

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

# Logistic Regression Pipeline
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Train 데이터로 학습
logistic_pipeline.fit(X_train, y_train)

# Validation 데이터의 취소 확률
val_prob = logistic_pipeline.predict_proba(X_val)[:, 1]

# 0.05 ~ 0.95까지 0.05 간격
thresholds = np.arange(0.05, 1.00, 0.05)

results = []

for threshold in thresholds:

    # Threshold 이상이면 취소(1)
    val_pred = (val_prob >= threshold).astype(int)

    results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision_score(y_val, val_pred, zero_division=0),
        "Recall": recall_score(y_val, val_pred, zero_division=0),
        "F1": f1_score(y_val, val_pred, zero_division=0)
    })

# 결과 표
threshold_results = pd.DataFrame(results)

display(threshold_results.round(3))

# F1 → Recall → Precision 순으로 가장 좋은 Threshold 선택
selected_row = threshold_results.sort_values(
    ["F1", "Recall", "Precision"],
    ascending=False
).iloc[0]

selected_threshold = selected_row["Threshold"]

print("선택된 Threshold:", selected_threshold)

,Threshold,Precision,Recall,F1
0,0.05,0.239,0.846,0.373
1,0.10,0.250,0.769,0.377
2,0.15,0.290,0.692,0.409
3,0.20,0.320,0.615,0.421
4,0.25,0.263,0.385,0.312
5,0.30,0.250,0.308,0.276
6,0.35,0.308,0.308,0.308
7,0.40,0.300,0.231,0.261
8,0.45,0.300,0.231,0.261
9,0.50,0.375,0.231,0.286


선택된 Threshold: 0.2


### 결과 해석

STEP 8을 수행하는 이유는 **모델이 계산한 “취소 확률”을 실제 0/1 판단으로 바꾸는 기준값을 정하기 위해서**입니다.

예를 들어 어떤 주문의 취소 확률이 `0.42`라고 해보면,

- Threshold가 `0.5`이면 → `0.42 < 0.5`이므로 **정상 주문(0)**
- Threshold가 `0.3`이면 → `0.42 ≥ 0.3`이므로 **취소 위험 주문(1)**

으로 판단이 달라집니다.

즉, **같은 모델이어도 Threshold를 어떻게 정하느냐에 따라 결과가 달라집니다.** 그래서 Test 데이터를 보기 전에 Validation 데이터에서 기준값을 먼저 정해야 합니다. :chatgpt-content-reference{index="0"}

이번 결과에서는 다음과 같이 나타났습니다.

- `0.3` → Recall `0.308`, F1 `0.276`
- `0.4` → Recall `0.231`, F1 `0.261`
- `0.5` → Precision `0.375`, Recall `0.231`, F1 `0.286`
- `0.6` → Recall `0.077`, F1 `0.111`
- `0.7` → Recall `0.077`, F1 `0.125`

가장 쉽게 보면 **Threshold를 낮추면 취소라고 판단하는 주문이 많아지고, Threshold를 높이면 취소라고 판단하는 주문이 줄어듭니다.** :chatgpt-content-reference{index="1"}

예를 들어 취소 주문이 13건 있다고 생각하면, 이번 Validation 결과에서 Recall이 `0.308`인 Threshold 0.3은 대략 **13건 중 4건 정도를 잡아낸 것**이고, Recall이 `0.077`인 Threshold 0.6~0.7은 **13건 중 1건 정도만 잡아낸 것**이라고 이해하면 됩니다.

이번에는 Threshold `0.5`에서 F1이 **0.286으로 가장 높았습니다.** Precision과 Recall의 균형이 현재 후보 중 가장 좋았다는 뜻입니다.

따라서 이 실습에서는 **Threshold = 0.5를 선택**합니다.

다만 중요한 시사점은, 현재 모델이 아주 강한 모델은 아니라는 점입니다. Threshold를 0.5로 정해도 Recall이 `0.231`이므로 실제 취소 주문 중 상당수를 놓치고 있습니다.

쉽게 말하면, **“취소 주문을 어느 정도 구분은 하지만 아직 많이 놓치는 모델”**이라고 볼 수 있습니다.

다음 STEP에서는 이제 모델과 Threshold를 더 이상 바꾸지 않고, **완전히 남겨둔 Test 데이터에서 최종 성능을 확인**하면 됩니다.